In [95]:
# Diagnostic cell — run this in one cell to confirm envs and test AzureChatOpenAI
from dotenv import load_dotenv
import os, traceback
from langchain_openai import AzureChatOpenAI 
from typing import TypedDict,Annotated
from langgraph.graph import StateGraph,START,END
import operator

load_dotenv()  # ensure .env loaded

print("AZURE_OPENAI_KEY present:", bool(os.getenv("AZURE_OPENAI_KEY")))
print("AZURE_OPENAI_ENDPOINT:", os.getenv("AZURE_OPENAI_ENDPOINT"))
print("AZURE_OPENAI_DEPLOYMENT:", os.getenv("AZURE_OPENAI_DEPLOYMENT"))
print("AZURE_OPENAI_API_VERSION:", os.getenv("AZURE_OPENAI_API_VERSION"))
print("OPENAI_API_KEY (if set):", bool(os.getenv("OPENAI_API_KEY")))

# Map env names the SDK might look for (non-destructive)
if os.getenv("AZURE_OPENAI_KEY") and not os.getenv("AZURE_OPENAI_API_KEY"):
    os.environ["AZURE_OPENAI_API_KEY"] = os.getenv("AZURE_OPENAI_KEY")
if os.getenv("AZURE_OPENAI_KEY") and not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.getenv("AZURE_OPENAI_KEY")

# Instantiate explicitly with api_key to avoid env name issues
try:
    model = AzureChatOpenAI(
        azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),
        temperature=0.2,
        api_key=os.getenv("AZURE_OPENAI_KEY"),  # explicit
    )
    print('AzureChatOpenAI instantiated OK')
    
except Exception as e:
    print('Failed to instantiate AzureChatOpenAI:')
    traceback.print_exc()

# Try a short call (small tokens) to test end-to-end
try:
    resp = model.invoke("Say hello in one short sentence.")
    print("Invoke result:", getattr(resp, "content", None) or resp)
except Exception as e:
    print('Error during invoke:')
    traceback.print_exc()

AZURE_OPENAI_KEY present: True
AZURE_OPENAI_ENDPOINT: https://crr-openai-esg.openai.azure.com/
AZURE_OPENAI_DEPLOYMENT: gpt-4o-mini
AZURE_OPENAI_API_VERSION: 2024-12-01-preview
OPENAI_API_KEY (if set): True
AzureChatOpenAI instantiated OK
Invoke result: Hello!


In [96]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict, Literal

In [97]:
class QuadState(TypedDict):
    a : int
    b : int
    c : int

    equation :str
    discriminant : str
    result : str

In [ ]:
def show_equation(state: QuadState):
    equation = f"{state['a']}X2{state['b']}X{state["c"]}"
    return { 'equation':equation} 

In [99]:
def calculate_discriminent(state:QuadState):
    d  = state['b']**2 - 4*state['a']*state['c']
    return {'discriminant':d}

In [100]:
def real_roots(state: QuadState):
    root1= (state['b']-(state['b']**2-4*state['a']*state['c'])**(0.5))/(2*state['a'])
    root2=(state['b']+(state['b']**2-4*state['a']*state['c'])**(0.5))/(2*state['a'])
    result = f'The root are {root1} and {root2}'
    return {'result':result}

In [101]:
def repeated_roots(state:QuadState):
    root = state['b']/(2*state['a'])
    result = f'The repeated_roots are {root}'
    return {'result':result}


In [102]:
def no_real_roots(state:QuadState):
    result = f'No real roots'
    return {'result':result}

In [103]:
def check_condition(state:QuadState) -> Literal['real_roots','repeated_roots','no_real_roots']:
    if state['discriminant']>0:
        return "real_roots"
    elif state['discriminant']==0:
        return "repeated_roots"
    else :
        return "no_real_roots"

In [104]:
graph = StateGraph(QuadState)

graph.add_node('show_equation', show_equation)
graph.add_node('calculate_discriminent', calculate_discriminent)
graph.add_node('real_roots', real_roots)
graph.add_node('repeated_roots', repeated_roots)
graph.add_node('no_real_roots', no_real_roots)


graph.add_edge(START, 'show_equation')
graph.add_edge('show_equation', 'calculate_discriminent')
graph.add_conditional_edges('calculate_discriminent',check_condition)
graph.add_edge('real_roots',END)
graph.add_edge('repeated_roots',END)
graph.add_edge('no_real_roots',END)

workflow = graph.compile()

In [106]:
initial_state ={
    'a':4,
    'b':-5,
    'c':-4
}
workflow.invoke(initial_state)

{'a': 4,
 'b': -5,
 'c': -4,
 'equation': '4X2-5X-4',
 'discriminant': 89,
 'result': 'The root are -1.8042476415070754 and 0.5542476415070754'}